# **NLP**

1. Natural Language Processing (NLP) is a subfield of Artificial Intelligence focused on enabling machines to understand, interpret, generate, and interact using human language

2. Types:
- Stateless: Learns on random portions of text at each iteration, without any information on the rest of the text.
- Statefull: Preserves the hidden state between training iterations and continues reading where it left off, allowing it to learn longer patterns

# **Generating Shakespearean Text Using a Character RNN**

In [3]:
import tensorflow as tf

shakespeare_url = "https://homl.info/shakespeare"
filepath = tf.keras.utils.get_file("shakespeare.txt", shakespeare_url)
with open(filepath) as f:
    shakespeare_text = f.read()

In [6]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [ ]:
# all distinct charecters 
"".join(sorted(set(shakespeare_text.lower())))

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [8]:
text_vec_layer = tf.keras.layers.TextVectorization(split="character", standardize="lower")
text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]

In [10]:
encoded -= 2  # drop tokens 0 (pad) and 1 (unknown), which we will not use
n_tokens = text_vec_layer.vocabulary_size() - 2  # number of distinct chars = 39
dataset_size = len(encoded)  # total number of chars = 1,115,394

In [11]:
n_tokens

39

In [12]:
dataset_size

1115394

In [13]:
def to_dataset(seq, len, shuffle=False, seed=None, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(seq)
    ds = ds.window(len + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda window_ds:window_ds.batch(len + 1))
    if shuffle:
        ds = ds.shuffle(100_000, seed=seed)
    ds = ds.batch(batch_size)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

In [19]:
# extra code – a simple example using to_dataset()
# There's just one sample in this dataset: the input represents "to b" and the output represents "o be"
list(to_dataset(text_vec_layer(["To be"])[0], len=4))

[(<tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[ 4,  5,  2, 23]], dtype=int64)>,
  <tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[ 5,  2, 23,  3]], dtype=int64)>)]

In [21]:
length = 100
tf.random.set_seed(42)
train_set = to_dataset(encoded[:1_000_000], len=length, shuffle=True,
                       seed=42)
valid_set = to_dataset(encoded[1_000_000:1_060_000], len=length)
test_set = to_dataset(encoded[1_060_000:], len=length)

In [23]:
train_set

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, None), dtype=tf.int64, name=None), TensorSpec(shape=(None, None), dtype=tf.int64, name=None))>

## **Building and Training the Char-RNN Model**